# 2. Embeddings

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
v1.dot(dv)

np.float32(0.32332397)

In [4]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730445)

# 3. Embedding Dataset

In [5]:
from src import FaqHttpLoader

loader = FaqHttpLoader()
documents = loader.load()
print(f"Loaded {len(documents)} documents")

Loaded 1208 documents


In [6]:
documents[10]

{'question': 'Do I need to enroll in the course before submitting homework?',
 'text': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.',
 'section': 'General Course-Related Questions',
 'course': 'machine-learning-zoomcamp'}

Generating embeddings

In [7]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [8]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [9]:
import numpy as np
X = np.array(vectors)
print(X.shape)

(1208, 384)


# 4. Vector Search

In [10]:
query = 'Can I still join the course after the start date?'
v_query = model.encode(query)

In [11]:
scores = X.dot(v_query)
# Best match
idx = np.argmax(scores)

print(f"Q: {query}")
print(f"Document [score={scores[idx]:.2%}]:\n{documents[idx]}")

Q: Can I still join the course after the start date?
Document [score=76.29%]:
{'question': 'Course: Can I still join the course after the start date?', 'text': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'section': 'General Course-Related Questions', 'course': 'data-engineering-zoomcamp'}


Top 5 results

In [12]:
top5 = np.argsort(-scores)[:5]
top5, scores[top5].round(3)

(array([553, 955,  29, 472, 558]),
 array([0.763, 0.758, 0.719, 0.654, 0.56 ], dtype=float32))

In [13]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'question': 'Course: Can I still join the course after the start date?', 'text': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'section': 'General Course-Related Questions', 'course': 'data-engineering-zoomcamp'}

0.7579371
{'question': 'Course - Can I still join the course after the start date?', 'text': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'section': 'General Course-Related Questions', 'course': 'mlops-zoomcamp'}

0.71921307
{'question': 'The course has already started. Can I still join it?', 'text': 'Yes, you can. Even though you missed the start date, you can

# 5. Vector Search with minsearch

In [14]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [15]:
# Searching
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vindex.search(
    query_vector,
    num_results=5
    )

In [16]:
results[0]

{'question': 'I just discovered the course. Can I still join?',
 'text': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'section': 'General Course-Related Questions',
 'course': 'llm-zoomcamp'}

In [17]:
# Filtering by course
results = vindex.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

# 6. RAG with Vector Search

First, create the OllamaClient client:

In [18]:
from dotenv import load_dotenv
from src import OllamaClient

load_dotenv()
ollama_client = OllamaClient()

Next, download and index the data:

In [19]:
from src import FaqHttpLoader, MinsearchIndex

loader = FaqHttpLoader()
documents = loader.load()
print(f"Loaded {len(documents)} documents")

index = MinsearchIndex(documents)

Loaded 1208 documents


Then use the RAGBase class:

In [20]:
from src import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=ollama_client,
    llm_model="granite4.1:3b",
)

query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, you can still sign up for the program. The context indicates that even if you discover the course after it has started, you are still eligible to join and participate. However, note that there will be deadlines for submitting final projects, so plan accordingly.'

This uses keyword search. Now let's swap it for vector search.

In [21]:
class RAGVector(RAGBase):
    def __init__(self, embeeder, **kwargs):
        super().__init__(**kwargs)
        self.embeeder = embeeder
    def search(self, query, n_results=5):
        query_vector = self.embeeder.encode(query)
        filter_dict = {"course": self._course_filter} if self._course_filter else {}

        results = self._index.search(
            query_vector,
            num_results=n_results,
            filter_dict=filter_dict
        )
        return results


In [22]:
vector_assistant = RAGVector(
    embeeder=model,
    index=vindex,
    llm_client=ollama_client,
    llm_model="granite4.1:3b",
    course_filter="data-engineering-zoomcamp"
)

In [23]:
vector_assistant.rag('the program has already begun, can I still sign up?')


'Yes, you can still join the course even after it has started. The context indicates that participation is open to anyone interested in joining, regardless of when they register. There will be deadlines for homework submissions and final projects, so plan accordingly. Additionally, all materials will remain available for review after the course concludes, allowing you to continue learning at your own pace.'

# 7. Vector Search with sqlitesearch

In [37]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [38]:
vs_index.fit(vectors, documents)

Searching

In [ ]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    num_results=5
    )
results

[{'question': 'I just discovered the course. Can I still join?',
  'text': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'section': 'General Course-Related Questions',
  'course': 'llm-zoomcamp'},
 {'question': 'The course has already started. Can I still join it?',
  'text': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'section': 'General Course-Related Questions',
  'course': 'machine-learning-zoomcamp'},
 {'question': 'Course - Can I still join the course after the start date?',
  'text': "Yes, even if you don't re

Filtering by course

In [41]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)
results

[{'question': 'I just discovered the course. Can I still join?',
  'text': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'section': 'General Course-Related Questions',
  'course': 'llm-zoomcamp'},
 {'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'text': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'section': 'General Course-Related Questions',
  'course': 'llm-zoomcamp'},
 {'question': 'When will the course be offered next?',
  'text': 'Summer 2025.',
  'section': 'General Course-Related Questions',
  'course': 'llm-zoomcamp'},
 {'question': 'Cour

In [42]:
vs_index.close()

Reopening the index

In [43]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer('all-MiniLM-L6-v2')

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [44]:
query_vector = model.encode('How do I run Kafka?')
results = vs_index.search(query_vector, num_results=5)
results

[{'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'text': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```',
  'section': 'Module 7: Streaming',
  'course': 'data-engineering-zoomcamp'},
 {'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'text': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CONFIG` in the scripts located at `src/main/java/org/example/` (e.g., `JsonConsumer.java`, `JsonProducer.java

In [45]:
vs_index.close()